# WorldQuant Alpha 101 多周期评价框架
### 指标好坏评价标准：
1. **Rank IC (相关性)**: 
   - 绝对值 **> 0.02**：因子有效。
   - 绝对值 **> 0.05**：因子非常优秀，具有很强的预测能力。
2. **IC IR (稳定性)**: 
   - **> 0.5**：因子表现非常稳健，预测能力的波动较小。
3. **Spread% (多空收益差)**:
   - 数值越大，说明因子区分牛股和熊股的能力越强。
4. **Q1_Ret%**: 最优组（因子值最大）的平均前向收益。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

FILE_PATH = r"F:\\self_quant\\data\\data\\all_stocks_merged_fixed.parquet"

print("正在加载全市场数据...")
df = pd.read_parquet(FILE_PATH)
df['date'] = pd.to_datetime(df['date'])

# 过滤主板，剔除ST和停牌
main_board_mask = df['code'].str.contains(r'^sh\.60|^sz\.00', regex=True)
df = df[main_board_mask].copy()
df = df[(df['tradestatus'] == 1) & (df['isST'] == 0)].copy()

print("计算多周期前向收益...")
df.sort_values(['code', 'date'], inplace=True)
df['fwd_ret_1d'] = df.groupby('code')['close'].shift(-1) / df['close'] - 1
df['fwd_ret_1w'] = df.groupby('code')['close'].shift(-5) / df['close'] - 1
df['fwd_ret_1m'] = df.groupby('code')['close'].shift(-20) / df['close'] - 1

print("构建量价矩阵 (Panel Data)...")
# 统一索引和列，确保计算时对齐
all_dates = sorted(df['date'].unique())
all_codes = sorted(df['code'].unique())

def get_matrix(col_name):
    return df.pivot(index='date', columns='code', values=col_name).reindex(index=all_dates, columns=all_codes)

prices = {
    'open': get_matrix('open'),
    'close': get_matrix('close'),
    'high': get_matrix('high'),
    'low': get_matrix('low'),
    'volume': get_matrix('volume'),
    'amount': get_matrix('amount')
}
prices['vwap'] = (prices['amount'] / (prices['volume'] * 100 + 1e-6)).replace([np.inf, -np.inf], np.nan)
prices['vwap'].fillna(prices['close'], inplace=True)

fwd_rets = {
    '1d': get_matrix('fwd_ret_1d'),
    '1w': get_matrix('fwd_ret_1w'),
    '1m': get_matrix('fwd_ret_1m')
}
print(f"矩阵构建完成，日期点数: {len(all_dates)}, 股票支数: {len(all_codes)}")

正在加载全市场数据...
计算多周期前向收益...
构建量价矩阵 (Panel Data)...
矩阵构建完成，日期点数: 3935, 股票支数: 3401


In [2]:
def rank(df): return df.rank(axis=1, pct=True)
def delay(df, d): return df.shift(d)
def delta(df, d): return df.diff(d)
def correlation(df1, df2, d): return df1.rolling(window=d).corr(df2)
def ts_min(df, d): return df.rolling(window=d).min()
def ts_max(df, d): return df.rolling(window=d).max()
def sign(df): return np.sign(df)

In [3]:
def alpha_002(p):
    v1 = rank(delta(np.log(p['volume'] + 1), 2))
    v2 = rank((p['close'] - p['open']) / p['open'])
    return -1 * correlation(v1, v2, 6)

def alpha_003(p): return -1 * correlation(rank(p['open']), rank(p['volume']), 10)

def alpha_005(p):
    return rank(p['open'] - p['vwap'].rolling(10).mean()) * (-1 * np.abs(rank(p['close'] - p['vwap'])))

def alpha_006(p): return -1 * correlation(p['open'], p['volume'], 10)

def alpha_009(p):
    d_close = delta(p['close'], 1)
    cond1 = ts_min(d_close, 5) > 0
    cond2 = ts_max(d_close, 5) < 0
    res = np.where(cond1, d_close, np.where(cond2, d_close, -d_close))
    return pd.DataFrame(res, index=p['close'].index, columns=p['close'].columns)

def alpha_012(p): return sign(delta(p['volume'], 1)) * (-1 * delta(p['close'], 1))

def alpha_041(p): return np.sqrt(p['high'] * p['low']) - p['vwap']

def alpha_054(p):
    num = -1 * (p['low'] - p['close']) * (p['open'] ** 5)
    den = (p['low'] - p['high']) * (p['close'] ** 5 + 1e-6)
    return num / den.replace(0, np.nan)

def alpha_101(p): return (p['close'] - p['open']) / ((p['high'] - p['low']) + 0.001)

alphas = {'Alpha_002': alpha_002, 'Alpha_003': alpha_003, 'Alpha_005': alpha_005, 'Alpha_006': alpha_006, 
          'Alpha_009': alpha_009, 'Alpha_012': alpha_012, 'Alpha_041': alpha_041, 'Alpha_054': alpha_054, 'Alpha_101': alpha_101}

In [4]:
def evaluate_multi_period(factor_panel, fwd_rets_dict, name):
    f_series = factor_panel.stack().rename('factor')
    if f_series.empty: return []
    
    res_list = []
    for period, ret_panel in fwd_rets_dict.items():
        r_series = ret_panel.stack().rename('fwd_ret')
        temp = pd.concat([f_series, r_series], axis=1).dropna()
        
        if temp.empty: 
            print(f"警告: {name} 在 {period} 周期对齐后无有效数据。")
            continue
        
        # 计算 IC
        daily_ic = temp.groupby('date').apply(lambda x: x['factor'].corr(x['fwd_ret'], method='spearman') if len(x)>10 else np.nan).dropna()
        if daily_ic.empty: continue
        
        ic_mean = daily_ic.mean()
        ic_ir = ic_mean / daily_ic.std() if daily_ic.std() != 0 else 0
        
        # 分组收益
        temp['quintile'] = temp.groupby('date')['factor'].transform(lambda x: pd.qcut(x.rank(method='first'), 5, labels=['Q5', 'Q4', 'Q3', 'Q2', 'Q1']))
        q_ret = temp.groupby('quintile')['fwd_ret'].mean() * 100
        
        res_list.append({
            'Factor': name, 'Period': period, 'Rank IC': ic_mean, 'IC IR': ic_ir,
            'Q1_Ret%': q_ret.get('Q1', 0),'Q2_Ret%': q_ret.get('Q2', 0),'Q3_Ret%': q_ret.get('Q3', 0),'Q4_Ret%': q_ret.get('Q4', 0), 'Q5_Ret%': q_ret.get('Q5', 0), 'Spread%': q_ret.get('Q1', 0) - q_ret.get('Q5', 0)
        })
    return res_list

In [5]:
all_results = []
for name, func in alphas.items():
    print(f"正在计算 {name}...", end=' ')
    try:
        factor_panel = func(prices)
        res = evaluate_multi_period(factor_panel, fwd_rets, name)
        if res:
            all_results.extend(res)
            print("完成")
        else:
            print("无有效IC结果")
    except Exception as e: 
        print(f"出错: {e}")

if not all_results:
    print("\n错误: 没有任何因子产生有效的测试结果。请检查数据对齐和因子公式。")
else:
    df_final = pd.DataFrame(all_results)
    
    for p in ['1d', '1w', '1m']:
        if p in df_final['Period'].values:
            print(f"\n=== 前向 {p} 收益测试结果摘要 ===")
            sub = df_final[df_final['Period'] == p].drop(columns='Period').set_index('Factor')
            display(sub.style.background_gradient(cmap='RdYlGn', subset=['Rank IC', 'IC IR', 'Spread%']).format(precision=4))
        else:
            print(f"\n=== 前向 {p} 收益无有效数据 ===")

正在计算 Alpha_002... 完成
正在计算 Alpha_003... 完成
正在计算 Alpha_005... 完成
正在计算 Alpha_006... 完成
正在计算 Alpha_009... 完成
正在计算 Alpha_012... 完成
正在计算 Alpha_041... 完成
正在计算 Alpha_054... 完成
正在计算 Alpha_101... 完成

=== 前向 1d 收益测试结果摘要 ===


,Rank IC,IC IR,Q1_Ret%,Q2_Ret%,Q3_Ret%,Q4_Ret%,Q5_Ret%,Spread%
Factor,,,,,,,,
Alpha_002,0.0239,0.3995,0.0446,0.0228,0.0230,0.0257,0.0252,0.0195
Alpha_003,0.0193,0.3234,0.0645,0.0494,0.0397,0.0212,-0.0339,0.0984
Alpha_005,0.0141,0.0965,0.0692,0.0486,0.0339,0.0146,-0.0255,0.0947
Alpha_006,0.0154,0.1822,0.0563,0.0533,0.0445,0.0294,-0.0427,0.0991
Alpha_009,0.0117,0.0983,-0.0679,0.0505,0.0745,0.0416,0.0593,-0.1272
Alpha_012,0.0210,0.2919,0.0346,0.0447,0.0524,0.0222,0.0041,0.0304
Alpha_041,-0.0135,-0.0934,-0.0177,0.0217,0.0385,0.0523,0.0710,-0.0887
Alpha_054,0.0162,0.1421,-0.0250,0.0440,0.0463,0.0308,-0.0007,-0.0243
Alpha_101,-0.0213,-0.1680,0.0510,0.0219,0.0569,0.0557,-0.0197,0.0707



=== 前向 1w 收益测试结果摘要 ===


,Rank IC,IC IR,Q1_Ret%,Q2_Ret%,Q3_Ret%,Q4_Ret%,Q5_Ret%,Spread%
Factor,,,,,,,,
Alpha_002,0.0244,0.4517,0.2225,0.1775,0.1608,0.1579,0.1306,0.0919
Alpha_003,0.0380,0.6672,0.3234,0.2631,0.2198,0.1329,-0.0770,0.4004
Alpha_005,0.0254,0.1588,0.3849,0.2808,0.2045,0.0997,-0.1079,0.4928
Alpha_006,0.0319,0.4000,0.2899,0.2555,0.2282,0.1720,-0.0835,0.3734
Alpha_009,0.0170,0.1528,0.0155,0.3088,0.3776,0.2561,-0.0368,0.0523
Alpha_012,0.0362,0.5284,0.2287,0.3080,0.2848,0.1692,-0.0697,0.2983
Alpha_041,-0.0258,-0.1623,-0.0913,0.1319,0.2223,0.2951,0.3943,-0.4857
Alpha_054,0.0135,0.1328,0.1231,0.2391,0.2338,0.1539,-0.0401,0.1633
Alpha_101,-0.0228,-0.1976,-0.0081,0.2074,0.3081,0.3028,0.1423,-0.1503



=== 前向 1m 收益测试结果摘要 ===


,Rank IC,IC IR,Q1_Ret%,Q2_Ret%,Q3_Ret%,Q4_Ret%,Q5_Ret%,Spread%
Factor,,,,,,,,
Alpha_002,0.0211,0.4128,0.7795,0.7114,0.6602,0.6397,0.5460,0.2334
Alpha_003,0.0462,0.8545,1.0350,0.8848,0.7571,0.5655,0.1488,0.8862
Alpha_005,0.0556,0.3250,1.4436,1.1246,0.8255,0.3984,-0.4011,1.8447
Alpha_006,0.0373,0.4858,0.9343,0.8549,0.7796,0.6452,0.1767,0.7575
Alpha_009,0.0054,0.0543,0.0562,1.0151,1.2410,0.9670,0.0602,-0.0040
Alpha_012,0.0381,0.5863,0.5614,1.1076,1.0956,0.7441,-0.1692,0.7306
Alpha_041,-0.0571,-0.3364,-0.4618,0.4119,0.8336,1.1394,1.4571,-1.9189
Alpha_054,0.0007,0.0073,0.4852,0.7391,0.7633,0.6665,0.3416,0.1436
Alpha_101,-0.0123,-0.1201,0.3097,0.7571,0.9260,0.8640,0.5239,-0.2142
